In [2]:
import sqlite3
import os

user_base = os.path.join(os.environ["USERPROFILE"], "myenv", "yanai-gaia-p")
db_path = os.path.join(user_base, "db", "output.db")
table_name = "result_table"

# ✅ 削除対象カラム
columns_to_delete = {"機ステータス回転設定値"}

conn = sqlite3.connect(db_path)
cur = conn.cursor()

# ✅ 現在のカラム一覧取得
cur.execute(f"PRAGMA table_info({table_name});")
all_columns_info = cur.fetchall()
all_columns = [row[1] for row in all_columns_info]

# ✅ 残すカラムのみ抽出
keep_columns = [col for col in all_columns if col not in columns_to_delete]

if not keep_columns:
    raise Exception("全カラム削除は不可です。keep_columnsが空です。")

columns_str = ", ".join(f"`{col}`" for col in keep_columns)

# ✅ 一時テーブル作成
cur.execute(f"CREATE TABLE {table_name}_backup AS SELECT {columns_str} FROM {table_name};")
print(f"[INFO] ✅ 一時テーブル {table_name}_backup 作成完了")

# ✅ 元テーブル削除
cur.execute(f"DROP TABLE {table_name};")
print(f"[INFO] ✅ 元テーブル {table_name} 削除完了")

# ✅ 新しいテーブルを元の名前にリネーム
cur.execute(f"ALTER TABLE {table_name}_backup RENAME TO {table_name};")
print(f"[INFO] ✅ 新しいテーブル {table_name} にリネーム完了")

conn.commit()
conn.close()

print("[INFO] ✅ 特定カラム削除完了")


OperationalError: unable to open database file